# 11장 — 에이전트의 성적표

웹 챕터(11장)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산합니다.
핵심은 pass@k의 **비편향 추정량**(코딩 벤치마크가 실제로 쓰는 그 공식)과 성적표의 **오차범위**예요.

> 🤖 **AI 튜터와 함께**: 아래 셀을 실행하면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch11')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. pass@k: 비편향 추정량

한 문제를 **n번** 시도해 **c번** 성공했다. 여기서 k개를 뽑아 한 번이라도 성공할 확률은?
실패 c'=n−c개 중 k개가 **전부** 뽑힐 확률을 1에서 뺀다(중복 없이 뽑으므로 조합):

$$\text{pass@}k = 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

소박한 공식 $1-(1-c/n)^k$ 은 매번 c/n로 복원 추출한다고 가정해 작은 표본에서 **편향**됩니다.

In [ ]:
from math import comb, sqrt

def passk(n, c, k):
    """비편향 pass@k = 1 - C(n-c, k) / C(n, k)."""
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def naive(n, c, k):
    """소박한(편향) 공식 = 1 - (1 - c/n)^k."""
    return 1.0 - (1 - c / n) ** k

n, c = 10, 4
print(f'pass@1 = {passk(n,c,1)*100:.1f}%   (= c/n = {c}/{n})')
print(f'pass@5(비편향) = {passk(n,c,5)*100:.2f}%')
print(f'pass@5(소박)   = {naive(n,c,5)*100:.2f}%')
print(f'  C(6,5)={comb(6,5)}, C(10,5)={comb(10,5)}, 6/252={comb(6,5)/comb(10,5):.4f}')

# 웹 챕터·StepQuiz 와 같은 값인지 자가검증
assert round(passk(10,4,1)*100, 1) == 40.0
assert round(passk(10,4,5)*100, 1) == 97.6
assert round(naive(10,4,5)*100, 1) == 92.2
assert comb(6,5) == 6 and comb(10,5) == 252
print('\n✅ pass@k 수치 자가검증 통과 (웹 실험실과 일치)')

## 2. k를 늘리면: 비편향 vs 소박한 공식

k가 커질수록 통과율은 오르지만, 두 공식이 벌어집니다. 실패가 6개뿐이라
비편향 공식은 k=7에서 이미 100%(6개를 다 뽑아도 7개째가 성공)인 반면, 소박한 공식은 계속 100% 밑입니다.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

ks = list(range(1, 9))
unb = [passk(10, 4, k) * 100 for k in ks]
nv = [naive(10, 4, k) * 100 for k in ks]
print('k       :', ks)
print('비편향  :', [round(x, 1) for x in unb])
print('소박    :', [round(x, 1) for x in nv])
assert [round(x, 1) for x in unb][:5] == [40.0, 66.7, 83.3, 92.9, 97.6]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(ks, unb, marker='o', color='#6366f1', label='비편향 pass@k')
ax.plot(ks, nv, marker='s', color='#94a3b8', label='소박한 공식 1-(1-c/n)^k')
ax.axhline(100, ls=':', lw=1, color='#cbd5e1')
ax.set_xlabel('k (뽑는 수)')
ax.set_ylabel('pass@k (%)')
ax.set_title('pass@k: 비편향 추정 vs 소박한 공식 (n=10, c=4)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 성적표의 오차범위: 과업 몇 개로 잰 점수인가

성적표도 표본입니다. 10부 6장의 냅킨 공식 **±≈98/√n %p** 를 그대로 적용합니다.
과업 수가 적으면 ±가 커서 두 에이전트를 구별할 수 없어요.

> 98의 출처: 오차범위 1.96×√(p(1−p)/n)×100%p 에서 흔들림이 가장 큰 p=50%를 넣으면
> √(p(1−p)) = 0.5 라 1.96×0.5×100 = **98**. "최악의 경우 오차"의 어림값입니다.
> 아래 출력의 상한(111.9)처럼 100%를 넘는 값은 확률이라 **100에서 잘라** 읽으세요.

In [ ]:
for nt in [20, 50, 200, 1000]:
    print(f'{nt:>4}건: ±{98/sqrt(nt):.1f}%p')

moe20 = 98 / sqrt(20)
print(f'\n20건 90% → 구간 [{90-moe20:.1f}, {90+moe20:.1f}]  (경쟁 70%를 덮음 → 우열 단정 불가)')

assert round(98/sqrt(20), 1) == 21.9
assert round(98/sqrt(200), 1) == 6.9
assert 90 - moe20 < 70  # 20건이면 90% 구간이 70%까지 내려온다
print('✅ 오차범위 수치 자가검증 통과')

## 4. 오늘의 발견

- **pass@k = 1 − C(n−c,k)/C(n,k)**: k번 안에 한 번 맞히기, 비편향 추정. pass@1=40%가 pass@5=97.6%로 뛴다.
- **소박한 공식은 작은 표본에서 편향**: 같은 4/10인데 92.2% vs 97.6%. “어떻게 셌나”가 성적표를 바꾼다.
- **성적표의 ±는 과업 수가 정한다**: 20건이면 ±21.9%p라 90%와 70%도 구별 못 한다. 200건이면 ±6.9%p.
- **궤적 평가·다중 비교**: 과정을 채점하면 병목이 보이고, 여러 벤치마크 중 최고만 발표하는 것은 골라 보여주기.

다음은 마지막 장(12장 캡스톤). 이 무기들을 전부 들고 가상 에이전트 제품의 도입 여부를 판정합니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**